In [8]:
#pip install seqeval

In [10]:
import random
import json
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizerFast, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
import torch
import numpy as np
from seqeval.metrics import accuracy_score, f1_score, classification_report


In [14]:

# -----------------------
# 1. Load your synthetic dataset
# -----------------------

with open("synthetic_vehicle_ner_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Extract unique labels and prepare BIO schema
unique_labels = set()
for item in data:
    for ent in item["entities"]:
        unique_labels.add(ent["label"])
labels = sorted(list(unique_labels))
bio_labels = ["O"] + [f"B-{l}" for l in labels] + [f"I-{l}" for l in labels]
label2id = {label: idx for idx, label in enumerate(bio_labels)}
id2label = {idx: label for label, idx in label2id.items()}



In [28]:
# -----------------------
# 2. Tokenize and align labels
# -----------------------
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def align_labels(text, entities):
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",  # ensure padding
        max_length=128,        # optional, set as needed
        return_offsets_mapping=True
    )
    
    labels = ["O"] * len(tokens["input_ids"])
    offsets = tokens["offset_mapping"]

    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        for idx, (offset_start, offset_end) in enumerate(offsets):
            if offset_start == start:
                labels[idx] = f"B-{label}"
            elif offset_start > start and offset_end <= end:
                labels[idx] = f"I-{label}"

    # Convert labels to IDs, pad or truncate to match token length
    label_ids = [label2id.get(label, label2id["O"]) for label in labels]
    tokens["labels"] = label_ids
    del tokens["offset_mapping"]
    return tokens



In [36]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)


In [18]:
# -----------------------
# 3. Prepare the dataset
# -----------------------
tokenized_data = [align_labels(item["text"], item["entities"]) for item in data]
dataset = Dataset.from_list(tokenized_data)
dataset = dataset.remove_columns(['offset_mapping'])



In [20]:
# -----------------------
# 4. Split into train/test
# -----------------------
split = dataset.train_test_split(test_size=0.2, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "test": split["test"]
})



In [22]:
# -----------------------
# 5. Load Model
# -----------------------
model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(bio_labels),
    id2label=id2label,
    label2id=label2id
)



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
# -----------------------
# 6. Define metrics
# -----------------------
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = [
        [id2label[label] for label in label_row if label != -100]
        for label_row in labels
    ]
    true_predictions = [
        [id2label[pred] for pred, label in zip(pred_row, label_row) if label != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]
    
    return {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "report": classification_report(true_labels, true_predictions)
    }



In [32]:
# -----------------------
# 7. Training
# -----------------------
args = TrainingArguments(
    output_dir="./ner-roberta-vehicle",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,  # <-- ADD THIS
    compute_metrics=compute_metrics
)


trainer.train()


/var/folders/hx/q1s4c6ws0dn84_dnnjvs7skc0000gp/T/ipykernel_13662/2412358275.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Report
1,0.001800,0.000187,1.000000,1.000000,precision recall f1-score support CustomerReportedSymptom 1.00 1.00 1.00 1073 DeviceProperties 1.00 1.00 1.00 1635 EquipmentName 1.00 1.00 1.00 1990 FaultCode 1.00 1.00 1.00 2000 FaultType 1.00 1.00 1.00 1638 MaintenanceMethod 1.00 1.00 1.00 2000 MeasurementValue 1.00 1.00 1.00 711 TimeDuration 1.00 1.00 1.00 1333 VehicleComponentLocation 1.00 1.00 1.00 2000 VehicleMakeModel 1.00 1.00 1.00 1853 micro avg 1.00 1.00 1.00 16233 macro avg 1.00 1.00 1.00 16233 weighted avg 1.00 1.00 1.00 16233
2,0.000200,0.000043,1.000000,1.000000,precision recall f1-score support CustomerReportedSymptom 1.00 1.00 1.00 1073 DeviceProperties 1.00 1.00 1.00 1635 EquipmentName 1.00 1.00 1.00 1990 FaultCode 1.00 1.00 1.00 2000 FaultType 1.00 1.00 1.00 1638 MaintenanceMethod 1.00 1.00 1.00 2000 MeasurementValue 1.00 1.00 1.00 711 TimeDuration 1.00 1.00 1.00 1333 VehicleComponentLocation 1.00 1.00 1.00 2000 VehicleMakeModel 1.00 1.00 1.00 1853 micro avg 1.00 1.00 1.00 16233 macro avg 1.00 1.00 1.00 16233 weighted avg 1.00 1.00 1.00 16233
3,0.000500,0.000032,1.000000,1.000000,precision recall f1-score support CustomerReportedSymptom 1.00 1.00 1.00 1073 DeviceProperties 1.00 1.00 1.00 1635 EquipmentName 1.00 1.00 1.00 1990 FaultCode 1.00 1.00 1.00 2000 FaultType 1.00 1.00 1.00 1638 MaintenanceMethod 1.00 1.00 1.00 2000 MeasurementValue 1.00 1.00 1.00 711 TimeDuration 1.00 1.00 1.00 1333 VehicleComponentLocation 1.00 1.00 1.00 2000 VehicleMakeModel 1.00 1.00 1.00 1853 micro avg 1.00 1.00 1.00 16233 macro avg 1.00 1.00 1.00 16233 weighted avg 1.00 1.00 1.00 16233
4,0.000200,0.000021,1.000000,1.000000,precision recall f1-score support CustomerReportedSymptom 1.00 1.00 1.00 1073 DeviceProperties 1.00 1.00 1.00 1635 EquipmentName 1.00 1.00 1.00 1990 FaultCode 1.00 1.00 1.00 2000 FaultType 1.00 1.00 1.00 1638 MaintenanceMethod 1.00 1.00 1.00 2000 MeasurementValue 1.00 1.00 1.00 711 TimeDuration 1.00 1.00 1.00 1333 VehicleComponentLocation 1.00 1.00 1.00 2000 VehicleMakeModel 1.00 1.00 1.00 1853 micro avg 1.00 1.00 1.00 16233 macro avg 1.00 1.00 1.00 16233 weighted avg 1.00 1.00 1.00 16233
5,0.000100,0.000018,1.000000,1.000000,precision recall f1-score support CustomerReportedSymptom 1.00 1.00 1.00 1073 DeviceProperties 1.00 1.00 1.00 1635 EquipmentName 1.00 1.00 1.00 1990 FaultCode 1.00 1.00 1.00 2000 FaultType 1.00 1.00 1.00 1638 MaintenanceMethod 1.00 1.00 1.00 2000 MeasurementValue 1.00 1.00 1.00 711 TimeDuration 1.00 1.00 1.00 1333 VehicleComponentLocation 1.00 1.00 1.00 2000 VehicleMakeModel 1.00 1.00 1.00 1853 micro avg 1.00 1.00 1.00 16233 macro avg 1.00 1.00 1.00 16233 weighted avg 1.00 1.00 1.00 16233


TrainOutput(global_step=5000, training_loss=0.011887278100848197, metrics={'train_runtime': 859.4337, 'train_samples_per_second': 46.542, 'train_steps_per_second': 5.818, 'total_flos': 1016353561217040.0, 'train_loss': 0.011887278100848197, 'epoch': 5.0})